# Gold: estrella comercial

Ejecuta `sql/gold/crm.sql`: `dim_account`, `dim_contact` + `fact_opportunity`, `fact_activity`, y `fact_lead` (mart independiente, sin FK -- ver `docs/decisiones.md`).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from pathlib import Path
from utils.db import get_psycopg2_connection, get_engine

engine = get_engine()
SQL_GOLD = Path("/home/jovyan/work/sql/gold")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

In [2]:
run_sql_file(SQL_GOLD / "crm.sql")

OK: crm.sql ejecutado


## 1. Conteos: gold vs. silver (deben coincidir exacto)

In [3]:
pd.read_sql("""
    SELECT 'dim_account' t, (SELECT count(*) FROM silver.crm__accounts) silver, (SELECT count(*) FROM gold.dim_account) gold
    UNION ALL SELECT 'dim_contact', (SELECT count(*) FROM silver.crm__contacts), (SELECT count(*) FROM gold.dim_contact)
    UNION ALL SELECT 'fact_opportunity', (SELECT count(*) FROM silver.crm__opportunities), (SELECT count(*) FROM gold.fact_opportunity)
    UNION ALL SELECT 'fact_activity', (SELECT count(*) FROM silver.crm__activities), (SELECT count(*) FROM gold.fact_activity)
    UNION ALL SELECT 'fact_lead', (SELECT count(*) FROM silver.crm__leads), (SELECT count(*) FROM gold.fact_lead)
""", engine)

,t,silver,gold
0,dim_account,5000,5000
1,dim_contact,15000,15000
2,fact_opportunity,3000,3000
3,fact_activity,20000,20000
4,fact_lead,2000,2000


## 2. Pregunta de negocio: tasa de cierre (`won`) por industria

In [4]:
pd.read_sql("""
    SELECT a.industry,
           count(*) AS oportunidades,
           count(*) FILTER (WHERE o.is_won) AS ganadas,
           round(100.0 * count(*) FILTER (WHERE o.is_won) / count(*) FILTER (WHERE NOT o.is_open), 1) AS pct_cierre_ganado,
           round(avg(o.amount) FILTER (WHERE o.is_won), 0) AS ticket_promedio_ganado
    FROM gold.fact_opportunity o
    JOIN gold.dim_account a ON a.account_id = o.account_id
    GROUP BY a.industry
    ORDER BY pct_cierre_ganado DESC
""", engine)

,industry,oportunidades,ganadas,pct_cierre_ganado,ticket_promedio_ganado
0,finance,414,62,68.9,38380.0
1,energy,242,38,62.3,50276.0
2,tech,567,89,61.8,34846.0
3,services,306,44,61.1,37613.0
4,manufacturing,341,50,61.0,32384.0
5,retail,453,84,59.2,41001.0
6,health,366,64,58.7,46437.0
7,education,311,45,57.0,28421.0


## 3. Pregunta de negocio: numero de actividades vs. resultado de la oportunidad

(¿mas contacto con el cliente se asocia a mas oportunidades ganadas?)

In [5]:
pd.read_sql("""
    SELECT o.stage,
           count(DISTINCT o.opportunity_id) AS oportunidades,
           round(count(act.activity_id)::numeric / count(DISTINCT o.opportunity_id), 1) AS actividades_promedio
    FROM gold.fact_opportunity o
    LEFT JOIN gold.fact_activity act ON act.opportunity_id = o.opportunity_id
    GROUP BY o.stage
    ORDER BY actividades_promedio DESC
""", engine)

,stage,oportunidades,actividades_promedio
0,prospect,621,3.5
1,qualification,611,3.4
2,lost,303,3.3
3,proposal,569,3.3
4,won,476,3.3
5,negotiation,420,3.2


## 4. Pregunta de negocio: conversion de leads por canal (`source`)

In [6]:
pd.read_sql("""
    SELECT source,
           count(*) AS leads,
           count(*) FILTER (WHERE is_converted) AS convertidos,
           round(100.0 * count(*) FILTER (WHERE is_converted) / count(*), 1) AS pct_conversion,
           round(avg(score), 1) AS score_promedio
    FROM gold.fact_lead
    GROUP BY source
    ORDER BY pct_conversion DESC
""", engine)

,source,leads,convertidos,pct_conversion,score_promedio
0,cold_call,204,30,14.7,48.9
1,referral,402,46,11.4,49.4
2,event,302,32,10.6,51.0
3,ads,282,28,9.9,50.1
4,web,810,69,8.5,50.4
